# QC Transfer Benchmark

Runs the hybrid-to-paired transfer benchmark with live progress output.

- Benchmark progress is printed split-by-split with elapsed time and ETA.
- Neural transfer prints per-epoch losses and ETA so you can see how long it will take.
- Use the run profiles below instead of the old smoke toggle.

In [1]:
from pathlib import Path
import sys

from IPython.display import display

repo_root = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from qc_framework import TransferBenchmarkConfig, QCTransferBenchmark

## Configuration

- `meaningful_baseline` is the safest first real run. It excludes the research neural family and focuses on the transfer baselines most likely to beat dummy.
- `neural_probe` keeps the baseline ladder and adds the neural transfer family with moderate epochs and live epoch logs.
- `full_research` is the expensive run once the baseline path is already stable.

In [2]:
PARQUET_PATH = repo_root / 'data/raw/33000_ROWS.parquet'
OUTPUT_DIR = repo_root / 'analysis/runs/transfer_notebook_xgb_context/outputs'

RUN_PROFILE = 'meaningful_baseline'

PROFILES = {
    'meaningful_baseline': {
        'model_families': ('dummy', 'paired_only', 'hybrid_only', 'hybrid_calibrated', 'hybrid_stack', 'hybrid_plus_paired', 'context_tabular'),
        'boosting_backends': ('lightgbm', 'catboost', 'xgboost'),
        'model_selection_splits': 3,
        'paired_weight_grid': (10.0, 25.0),
        'lightgbm_estimators': 120,
        'xgboost_estimators': 120,
        'neural_pretrain_epochs': 0,
        'neural_finetune_epochs': 0,
        'neural_batch_size': 128,
    },
    'neural_probe': {
        'model_families': ('dummy', 'paired_only', 'hybrid_only', 'hybrid_calibrated', 'hybrid_stack', 'hybrid_plus_paired', 'context_tabular', 'context_transfer'),
        'boosting_backends': ('lightgbm', 'catboost', 'xgboost'),
        'model_selection_splits': 2,
        'paired_weight_grid': (10.0, 25.0),
        'lightgbm_estimators': 120,
        'xgboost_estimators': 120,
        'neural_pretrain_epochs': 8,
        'neural_finetune_epochs': 5,
        'neural_batch_size': 128,
    },
    'full_research': {
        'model_families': ('dummy', 'paired_only', 'hybrid_only', 'hybrid_calibrated', 'hybrid_stack', 'hybrid_plus_paired', 'context_tabular', 'context_transfer'),
        'boosting_backends': ('lightgbm', 'catboost', 'xgboost'),
        'model_selection_splits': 5,
        'paired_weight_grid': (10.0, 25.0, 50.0),
        'lightgbm_estimators': 250,
        'xgboost_estimators': 250,
        'neural_pretrain_epochs': 35,
        'neural_finetune_epochs': 20,
        'neural_batch_size': 128,
    },
}

profile = PROFILES[RUN_PROFILE]

config = TransferBenchmarkConfig(
    parquet_path=PARQUET_PATH,
    paired_final_holdout_rows=100,
    model_selection_splits=profile['model_selection_splits'],
    model_families=profile['model_families'],
    boosting_backends=profile['boosting_backends'],
    paired_weight_grid=profile['paired_weight_grid'],
    lightgbm_estimators=profile['lightgbm_estimators'],
    xgboost_estimators=profile['xgboost_estimators'],
    neural_pretrain_epochs=profile['neural_pretrain_epochs'],
    neural_finetune_epochs=profile['neural_finetune_epochs'],
    neural_batch_size=profile['neural_batch_size'],
    verbose=True,
    neural_verbose=True,
)

print('RUN_PROFILE =', RUN_PROFILE)
config

RUN_PROFILE = meaningful_baseline


TransferBenchmarkConfig(parquet_path=PosixPath('/Users/paulruiz/Documents/Predicting_Good_Units/data/raw/33000_ROWS.parquet'), primary_targets=('fpos', 'fmiss'), secondary_targets=('accuracy',), source_fmiss_target='fmiss_extended', final_eval_fmiss_target='fmiss', paired_final_holdout_rows=100, allow_unlabeled_paired=True, feature_views=('unit_raw', 'norm_swap', 'shape_only', 'recording_relative', 'recording_context', 'study_identity'), model_families=('dummy', 'paired_only', 'hybrid_only', 'hybrid_calibrated', 'hybrid_stack', 'hybrid_plus_paired', 'context_tabular'), feature_view_sets=(('shape_only', ('shape_only',)), ('unit_raw', ('unit_raw',)), ('contextual_full', ('norm_swap', 'recording_relative', 'recording_context', 'study_identity'))), protocol_modes=('inductive', 'contextual'), model_selection_splits=3, model_selection_test_size=0.4, random_state=42, lightgbm_estimators=120, lightgbm_learning_rate=0.05, lightgbm_num_leaves=31, xgboost_estimators=120, xgboost_learning_rate=0.0

## Run

Execute the next cell and watch the output. In notebook mode you will see:

- current split / target / family
- elapsed time and benchmark ETA
- neural pretrain and finetune epoch losses with ETA

In [3]:
runner = QCTransferBenchmark(config)
artifacts = runner.run()

Transfer benchmark start | paired matched=207 | hybrid=24211 | paired=8982 | total tasks=192
[1/192] inductive model_selection split=1 target=fpos family=dummy ...
[1/192] inductive model_selection split=1 target=fpos family=dummy done in 0.0s | elapsed=0.0m | eta=0.3m
[2/192] inductive model_selection split=1 target=fpos family=ceiling ...
[2/192] inductive model_selection split=1 target=fpos family=ceiling done in 0.0s | elapsed=0.0m | eta=0.1m
[3/192] inductive model_selection split=1 target=fpos family=paired_only ...
[3/192] inductive model_selection split=1 target=fpos family=paired_only done in 0.6s | elapsed=0.0m | eta=0.7m
[4/192] inductive model_selection split=1 target=fpos family=hybrid_only ...
[4/192] inductive model_selection split=1 target=fpos family=hybrid_only done in 4.1s | elapsed=0.1m | eta=3.8m
[5/192] inductive model_selection split=1 target=fpos family=hybrid_calibrated ...
[5/192] inductive model_selection split=1 target=fpos family=hybrid_calibrated done in 2

## Review Results

In [6]:
display(artifacts['config'])
display(artifacts['feature_leakage_checks'])
display(artifacts['split_report'])
print('Per-target deployment winners (inductive default)')
display(artifacts['winner_summary'][['target', 'protocol_mode', 'candidate_id', 'model_family', 'model_name', 'feature_view', 'mae']].sort_values(['target']))
print('Best target-specific recommendation by protocol')
display(artifacts['per_target_recommendation'][['protocol_mode', 'target', 'candidate_id', 'model_family', 'model_name', 'feature_view', 'mae']].sort_values(['protocol_mode', 'target']))
print('Combined primary winner (diagnostic only)')
display(artifacts['combined_winner_summary'])
display(artifacts['backend_comparison'])
display(artifacts['model_selection_summary'].sort_values(['protocol_mode', 'target', 'mae', 'candidate_id']).head(40))
display(artifacts['final_holdout_summary'].sort_values(['protocol_mode', 'target', 'mae', 'candidate_id']).head(40))

,parquet_path,primary_targets,secondary_targets,source_fmiss_target,final_eval_fmiss_target,paired_final_holdout_rows,allow_unlabeled_paired,feature_views,model_families,feature_view_sets,...,neural_batch_size,neural_lr,neural_domain_loss_weight,neural_target_loss_weight,target_transform,deploy_best_per_target,use_recording_context,include_ceiling_diagnostics,verbose,neural_verbose
0,/Users/paulruiz/Documents/Predicting_Good_Unit...,"(fpos, fmiss)","(accuracy,)",fmiss_extended,fmiss,100,True,"(unit_raw, norm_swap, shape_only, recording_re...","(dummy, paired_only, hybrid_only, hybrid_calib...","((shape_only, (shape_only,)), (unit_raw, (unit...",...,128,0.001,0.15,2.0,logit,True,True,True,True,True


,feature_view,n_features,raw_normalized_overlap,passes_norm_swap_check
0,shape_only,70,0,1
1,unit_raw,164,13,1
2,contextual_full,746,0,1


,paired_matched_rows,target_final_holdout_rows,actual_final_holdout_rows,final_train_rows,final_holdout_recordings,final_train_recordings,recording_overlap
0,207,100,100,107,15,14,0


Per-target deployment winners (inductive default)


,target,protocol_mode,candidate_id,model_family,model_name,feature_view,mae
0,fmiss,inductive,inductive|paired_only|xgboost|shape_only,paired_only,xgboost,shape_only,0.253565
1,fpos,inductive,inductive|paired_only|lightgbm|unit_raw,paired_only,lightgbm,unit_raw,0.125948


...

Combined primary winner (diagnostic only)


,protocol_mode,candidate_id,model_family,model_name,feature_view,primary_avg_mae,beats_best_paired_only,advances_all_primary_targets,eligible_default,best_paired_only_primary_avg_mae
0,inductive,inductive|paired_only|lightgbm|unit_raw,paired_only,lightgbm,unit_raw,0.125948,False,False,True,0.125948


,protocol_mode,backend,mae,r2
0,contextual,lightgbm,0.156711,0.357893
2,contextual,xgboost,0.208512,-0.128277
1,contextual,other,0.224124,-0.001742
3,inductive,lightgbm,0.183021,0.228325
5,inductive,xgboost,0.212175,-0.233836
4,inductive,other,0.224124,-0.001742


,protocol_mode,candidate_id,model_family,model_name,feature_view,target,is_ceiling_model,mae,rmse,r2,...,primary_avg_mae,dummy_mae,best_hybrid_only_mae,beats_dummy,beats_best_hybrid_only,beats_dummy_all_primary_targets,beats_hybrid_only_all_primary_targets,advances_target,advances_all_primary_targets,advances
0,contextual,contextual|ceiling|recording_oracle_mean|ceiling,ceiling,recording_oracle_mean,ceiling,accuracy,True,0.200822,0.236362,0.340368,...,NaN,NaN,NaN,False,False,NaN,NaN,False,False,False
63,contextual,contextual|paired_only|xgboost|shape_only,paired_only,xgboost,shape_only,accuracy,False,0.218202,0.263906,0.160830,...,0.179954,NaN,NaN,False,False,True,True,False,True,False
57,contextual,contextual|paired_only|lightgbm|shape_only,paired_only,lightgbm,shape_only,accuracy,False,0.220065,0.274609,0.109580,...,0.191965,NaN,NaN,False,False,False,True,False,False,False
45,contextual,contextual|hybrid_plus_paired|xgboost_w25|cont...,hybrid_plus_paired,xgboost_w25,contextual_full,accuracy,False,0.235683,0.304787,-0.228335,...,0.208931,NaN,NaN,False,False,False,True,False,False,False
3,contextual,contextual|context_tabular|lightgbm|contextual...,context_tabular,lightgbm,contextual_full,accuracy,False,0.236932,0.299623,-0.075220,...,0.212179,NaN,NaN,False,False,False,True,False,False,False
54,contextual,contextual|paired_only|lightgbm|contextual_full,paired_only,lightgbm,contextual_full,accuracy,False,0.236932,0.299623,-0.075220,...,0.212179,NaN,NaN,False,False,False,True,False,False,False
39,contextual,contextual|hybrid_plus_paired|lightgbm_w25|con...,hybrid_plus_paired,lightgbm_w25,contextual_full,accuracy,False,0.243783,0.313398,-0.304237,...,0.232905,NaN,NaN,False,False,False,True,False,False,False
42,contextual,contextual|hybrid_plus_paired|xgboost_w10|cont...,hybrid_plus_paired,xgboost_w10,contextual_full,accuracy,False,0.246079,0.319831,-0.422920,...,0.213227,NaN,NaN,False,False,False,True,False,False,False
48,contextual,contextual|hybrid_stack|lightgbm_residual|cont...,hybrid_stack,lightgbm_residual,contextual_full,accuracy,False,0.246855,0.301770,-0.135630,...,0.231928,NaN,NaN,False,False,False,True,False,False,False
51,contextual,contextual|hybrid_stack|xgboost_residual|conte...,hybrid_stack,xgboost_residual,contextual_full,accuracy,False,0.250817,0.311371,-0.282484,...,0.216586,NaN,NaN,False,False,False,True,False,False,False


,protocol_mode,candidate_id,model_family,model_name,feature_view,target,is_ceiling_model,mae,rmse,r2,...,advances,advances_target,advances_all_primary_targets,beats_dummy_all_primary_targets,beats_hybrid_only_all_primary_targets,best_paired_only_primary_avg_mae,beats_best_paired_only,best_paired_only_target_mae,beats_best_paired_only_target,eligible_target_default
15,contextual,contextual|hybrid_stack|lightgbm_residual|cont...,hybrid_stack,lightgbm_residual,contextual_full,accuracy,False,0.174366,0.227440,0.451521,...,False,False,False,False,True,0.124521,False,0.237424,True,False
12,contextual,contextual|hybrid_plus_paired|xgboost_w25|cont...,hybrid_plus_paired,xgboost_w25,contextual_full,accuracy,False,0.192177,0.248435,0.345586,...,False,False,False,False,True,0.124521,False,0.237424,True,False
0,contextual,contextual|context_tabular|lightgbm|contextual...,context_tabular,lightgbm,contextual_full,accuracy,False,0.194366,0.254172,0.315014,...,False,False,False,False,True,0.124521,False,0.237424,True,False
8,contextual,contextual|hybrid_calibrated|xgboost+isotonic|...,hybrid_calibrated,xgboost+isotonic,contextual_full,accuracy,False,0.207714,0.252725,0.322789,...,False,False,False,False,True,0.124521,False,0.237424,True,False
19,contextual,contextual|paired_only|xgboost|shape_only,paired_only,xgboost,shape_only,accuracy,False,0.237424,0.314440,-0.048337,...,False,False,True,True,True,0.124521,False,0.237424,False,True
3,contextual,contextual|dummy|paired_mean|none,dummy,paired_mean,none,accuracy,False,0.275704,0.308629,-0.009951,...,False,False,False,False,True,0.124521,False,0.237424,False,False
9,contextual,contextual|hybrid_only|xgboost|contextual_full,hybrid_only,xgboost,contextual_full,accuracy,False,0.284610,0.375133,-0.492099,...,False,False,False,False,False,0.124521,False,0.237424,False,False
1,contextual,contextual|context_tabular|lightgbm|contextual...,context_tabular,lightgbm,contextual_full,fmiss,False,0.152413,0.199268,0.526737,...,False,False,False,False,True,0.124521,False,0.253565,True,False
16,contextual,contextual|hybrid_stack|xgboost_residual|conte...,hybrid_stack,xgboost_residual,contextual_full,fmiss,False,0.183232,0.259850,0.195226,...,False,False,False,False,True,0.124521,False,0.253565,True,False
13,contextual,contextual|hybrid_plus_paired|xgboost_w25|cont...,hybrid_plus_paired,xgboost_w25,contextual_full,fmiss,False,0.200646,0.260816,0.189232,...,False,False,False,False,True,0.124521,False,0.253565,True,False


## Save Artifacts

In [8]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, obj in artifacts.items():
    if hasattr(obj, 'to_csv'):
        obj.to_csv(OUTPUT_DIR / f'{name}.csv', index=False)

print('Saved to:', OUTPUT_DIR)

Saved to: /Users/paulruiz/Documents/Predicting_Good_Units/analysis/runs/transfer_notebook_xgb_context/outputs
